## Progettazione di un data warehouse
Un franchising di noleggio auto, con punti vendita in diverse città italiane, vuole realizzare un data warehouse per analizzare lo storico dei noleggi, l’utilizzo dei veicoli, i tragitti effettuati e il profilo degli utenti.

### Descrizione del problema
L’azienda dispone di un sistema informatico per la gestione operativa dei noleggi. Il database applicativo contiene le seguenti informazioni relative agli utenti, che possono essere aziende o persone fisiche, e ai punti vendita, gestiti da aziende indipendenti aderenti al franchising. La stessa azienda può gestire più punti vendita. Il sistema raccoglie inoltre informazioni sui veicoli, condivisi tra tutte le aziende aderenti al franchising, caratterizzati in base alla loro tipologia. Infine, il sistema raccoglie informazioni sui noleggi: registrando l’utente, il veicolo, il punto vendita di ritiro e quello di riconsegna, le date e le ore di inizio e fine, il metodo e il canale di pagamento, gli eventuali extra acquistati e il prezzo totale pagato dal cliente (pocket price), comprensivo di extra e sconti. Alla conclusione del noleggio vengono registrati anche i chilometri percorsi, il punto di riconsegna e la data e l’ora di fine noleggio (prima della fine del noleggio sono riportati la data e l’ora di fine stimata, dichiarata dal cliente in fase di prenotazione).

### Data Warehouse Design
Il data warehouse deve permettere di fare analisi sul ***numero di noleggi***, ***ricavi*** e ***chilometri percorsi*** in relazione alle seguenti analisi di interesse:
- Gli *utenti* che effettuano i noleggi, da profilare in relazione al fatto che siano o meno persone fisiche. Se sono persone fisiche, vi è un’ulteriore differenziazione tra chi ha meno di 25 anni e chi ha almeno 25 anni. Di conseguenza, le tipologie di utente sono le seguenti: *azienda, persona fisica con meno di 25 anni, persona fisica con almeno 25 anni*.

- Il *punto vendita* di *destinazione e arrivo*. Si vuole memorizzare anche la *città, provincia e regione* del punto di arrivo. Il sistema deve tenere traccia, per ciascun punto vendita, dell’*azienda* di cui fa parte.

- I veicoli noleggiati, mediante il *modello* (e.g., Panda), il *produttore* del modello (e.g., Fiat, Ford,
etc…), il *tipo di alimentazione* (che può essere benzina, diesel, GPL, ibrido o elettrico) e il *tipo di
veicolo* (ad esempio furgone, SUV, utilitaria, …).

- I periodi di tempo in cui i noleggi sono avvenuti (*mese, trimestre e anno*). Il mese di riferimento di un noleggio è il mese in cui inizia il noleggio. Un noleggio che si estende su due mesi viene quindi attribuito al mese in cui è iniziato. Lo stesso vale per i trimestri e per gli anni. Si considerano esclusivamente i noleggi conclusi.

#### Il data warehouse deve quindi supportare i seguenti KPI:
1. Fatturato e numero di noleggi trimestrale e annuale.

2. Fatturato e numero di noleggi per ogni azienda nel franchising.

3. Numero di noleggi per città di partenza e di arrivo per trimestre.

4. Ricavo e numero di noleggi in base al tipo di veicolo.

5. Chilometri percorsi totali per ogni mese e tipo di veicolo.

6. Tempo noleggio totale in ***ore*** per tipo di veicolo, mese, tipo di utente.

7. Separatamente per ogni mese, tipo di alimentazione del veicolo e regione di partenza, calcolare:
    - Il numero totale di ore di noleggio.
    - Il numero totale di noleggi rispetto al totale per ciascuna regione e tipo di alimentazione.
    - La media mensile dei chilometri percorsi per noleggio.
    - Il numero cumulativo di noleggi dall’inizio dell’anno fino alla fine del mese considerato per la regione e tipo di alimentazione.
    - La media mobile dei chilometri percorsi calcolata sul mese corrente e sui due mesi precedenti.

8. Separatamente per ogni trimestre, città di partenza e tipologia di utente (azienda, persona con meno di 25 anni, persona con almeno 25 anni), calcolare:
    - Il numero di noleggi.
    - La percentuale del fatturato trimestrale della città rispetto al totale della regione nello stesso trimestre, per ogni tipo di utente.
    - La posizione della città nella classifica regionale del trimestre suddivisa per tipo di utente, ordinata per fatturato decrescente

## Progettazione concettuale

### Facts:
- #### Noleggi

![Schema Concettuale](../Data%20Warehouse/img/03_DSTBD_ProgettazioneNoleggi.png)

## Progettazione Logica
**Noleggi** (*IdUtente*, *IdPuntoVenditaPartenza*, *IdPuntoVenditaArrivo*, *IdVeicolo*, *IdTempo*, Ricavo, NumNoleggi, kmTotali, OreTotali)

**Tempo** (*IdTempo*, Mese, Trimestre, Anno)

**Veicolo** (*IdVeicolo*, Modello, Produttore, TipoAlimentazione, TipoVeicolo)

**PuntoVendita** (*IdPuntoVendita*, PuntoVendita, Azienda, Città, Provincia, Regione)

**Utente** (*IdUtente*, TipoUtente)

## Query SQL

1. Fatturato e numero di noleggi trimestrale e annuale.

In [ ]:
SELECT T.Trimestre, T.Anno, SUM(N.Ricavo), SUM(N.NumNoleggi)
FROM Noleggi N
JOIN Tempo T ON T.IdTempo = N.IdTempo
GROUP BY T.Trimestre, T.Anno;

2. Fatturato e numero di noleggi per ogni azienda nel franchising.

In [ ]:
SELECT PV.Azienda, SUM(N.Ricavo), SUM(N.NumNoleggi)
FROM Noleggi N
JOIN PuntoVendita PV ON PV.IdPuntoVendita = N.IdPuntoVendita
GROUP BY PV.Azienda;

3. Numero di noleggi per città di partenza e di arrivo per trimestre.

In [ ]:
SELECT PP.Citta AS Partenza, PA.Citta AS Arrivo, T.Trimestre, T.Anno, SUM(N.NumNoleggi) 
FROM Noleggi N
JOIN PuntoVendita PP ON N.IdPuntoVenditaPartenza = PP.IdPuntoVendita
JOIN PuntoVendita PA ON N.IdPuntoVenditaArrivo = PA.IdPuntoVendita
JOIN Tempo T ON N.IdTempo = T.IdTempo
GROUP BY PP.Citta, PA.Citta, T.Trimestre, T.Anno;

4. Ricavo e numero di noleggi in base al tipo di veicolo.

In [ ]:
SELECT V.TipoVeicolo, SUM(N.Ricavo), SUM(N.NumNoleggi)
FROM Noleggi N 
JOIN Veicolo V ON V.IdVeicolo = N.IdVeicolo
GROUP BY V.TipoVeicolo;

5. Chilometri percorsi totali per ogni mese e tipo di veicolo.

In [ ]:
SELECT T.Mese, V.TipoVeicolo, SUM(N.kmTotali)
FROM Noleggi N 
JOIN Tempo T ON T.IdTempo = N.IdTempo
JOIN Veicolo V ON V.IdVeicolo = N.IdVeicolo
GROUP BY T.Mese, V.TipoVeicolo;

6. Tempo noleggio totale in ***ore*** per tipo di veicolo, mese, tipo di utente.

In [ ]:
SELECT V.TipoVeicolo, T.Mese, U.TipoUtente, SUM(N.OreTotali)
FROM Noleggi N 
JOIN Tempo T ON T.IdTempo = N.IdTempo
JOIN Veicolo V ON V.IdVeicolo = N.IdVeicolo
JOIN Utente U ON U.IdUtente = N.IdUtente
GROUP BY V.TipoVeicolo, T.Mese, U.TipoUtente

7. Separatamente per ogni mese, tipo di alimentazione del veicolo e regione di partenza, calcolare:
    - Il numero totale di ore di noleggio.
    - Il numero totale di noleggi rispetto al totale per ciascuna regione e tipo di alimentazione.
    - La media mensile dei chilometri percorsi per noleggio.
    - Il numero cumulativo di noleggi dall’inizio dell’anno fino alla fine del mese considerato per la regione e tipo di alimentazione.
    - La media mobile dei chilometri percorsi calcolata sul mese corrente e sui due mesi precedenti.

In [ ]:
SELECT T.Mese, T.Anno, V.TipoAlimentazione, PP.Regione AS RegionePartenza,
    SUM(N.OreTotali) AS TotOreNoleggio, SUM(N.NumNoleggi), SUM(N.kmTotali)
    SUM(SUM(N.NumNoleggi)) OVER (
        PARTITION BY PP.Regione, V.TipoAlimentazione
    ) AS TotNoleggioXRegioneTipoAlimentazione,
    (SUM(SUM(N.kmTotali)) / SUM(SUM(N.NumNoleggi))) OVER(
        PARTITION BY T.Mese
    ) AS AvgMensileKM,
    SUM(SUM(N.NumNoleggi)) OVER (
        PARTITION BY T.Anno, PP.Regione, V.TipoAlimentazione
        ORDER BY T.Mese ASC
        ROWS BETWEEN UNBOUNDED PRECDING AND CURRENT ROW
    ) AS CumulativoNoleggi,
    AVG(SUM(N.kmTotali)) / OVER (
        PARTITION BY T.Anno, PP.Regione, V.TipoAlimentazione
        ORDER BY T.Mese ASC
        ROWS BETWEEN 2 PRECDING AND CURRENT ROW
    ) AS MobileAvg
FROM Noleggi N 
JOIN Tempo T ON T.IdTempo = N.IdTempo
JOIN Veicolo V ON V.IdVeicolo = N.IdVeicolo
JOIN PuntoVendita PP ON PP.IdPuntoVendita = N.IdPuntoVendita
GROUP BY T.Mese, T.Anno, V.TipoAlimentazione, PP.Regione

8. Separatamente per ogni trimestre, città di partenza e tipologia di utente (azienda, persona con meno di 25 anni, persona con almeno 25 anni), calcolare:
    - Il numero di noleggi.
    - La percentuale del fatturato trimestrale della città rispetto al totale della regione nello stesso trimestre, per ogni tipo di utente.
    - La posizione della città nella classifica regionale del trimestre suddivisa per tipo di utente, ordinata per fatturato decrescente

In [ ]:
SELECT T.Trimestre,T.Anno, PA.Città AS Partenza, PA.Regione, U.TipoUtente, 
    SUM(N.NumNoleggi), SUM(N.Ricavo) AS Fatturato,
    100 * SUM(N.Ricavo) / SUM(SUM(N.Ricavo)) OVER (
        PARTITION BY PA.Regione, T.Trimestre, U.TipoUtente
    ) AS FatturatoTrimestrale,
    RANK() OVER (
        PARTITION BY PA.Regione, T.Trimestre, T.Anno U.TipoUtente
        ORDER BY SUM(N.Ricavo) DESC
    ) AS PosClassifica
FROM Noleggi N 
JOIN Tempo T ON T.IdTempo = N.IdTempo
JOIN Utente U ON U.IdUtente = N.IdUtente    
JOIN PuntoVendita PA ON PA.IdPuntoVendita = N.IdPuntoVendita
GROUP BY T.Trimestre, T.Anno, PA.Città, PA.Regione, U.TipoUtente

## Viste Materializzate in Oracle

Si considerino le seguenti query frequenti:

- Calcolare il numero di noleggi e media dei chilometri percorsi per noleggio, per trimestre, tipo di alimentazione e regione di partenza degli ultimi due anni.

- Considerando i noleggi di auto con produttore “Fiat”, calcolare il numero cumulativo di noleggi e media progressiva annuale dei chilometri per noleggio, per trimestre, tipo di alimentazione e regione di partenza.

- Calcolare il fatturato totale e fatturato medio per noleggio, per trimestre, tipo di alimentazione e regione di partenza.

- Calcolare il numero di noleggi e durata media in ore, per anno e tipo di alimentazione.

1. Definire mediante CREATE MATERIALIZED VIEW una vista materializzata, denominata VM1 , utile ad accelerare tutte le query indicate. Motivare la struttura scelta rispetto ai tempi di risposta e allo spazio occupato.

In [ ]:
CREATE MATERIALIZED VIEW VM1 AS
SELECT V.TipoAlimentazione, T.Trimestre, T.Anno, PP.Regione, V.Produttore,
    SUM(N.NumNoleggi) AS NumNoleggi, SUM(N.kmTotali) AS kmTotali, 
    SUM(N.Ricavo) AS FatturatoTotale, SUM(N.OreTotali) AS OreTotali
FROM Noleggio N 
JOIN Tempo T ON T.IdTempo = N.IdTempo
JOIN PuntoVendita PP ON PP.IdPuntoVendita = N.IdPuntoVendita
JOIN Veicolo V ON V.IdVeicolo = N.IdVeicolo
GROUP BY V.TipoAlimentazione, V.Produttore, T.Trimestre, T.Anno, PP.Regione

La vista materializzata VM1 è stata progettata considerando le quattro interrogazioni frequenti, selezionando gli attributi dimensionali e le misure necessari per soddisfarle.

In particolare, sono state mantenute le dimensioni Anno, Trimestre, TipoAlimentazione, Regione e Produttore, mentre sono state memorizzate le misure additive Numero di noleggi, Chilometri percorsi, Ricavo e Ore di noleggio.

La granularità della vista è definita dalla combinazione di tali attributi dimensionali.

Questa scelta consente di ridurre i tempi di risposta delle interrogazioni, evitando la ripetizione dei JOIN tra tabella dei fatti e dimensioni e limitando le operazioni di aggregazione sui dati originari. Le misure memorizzate permettono inoltre di calcolare successivamente medie, somme cumulative e medie progressive.

Dal punto di vista dello spazio occupato, la vista presenta un compromesso tra prestazioni e dimensioni: aggregando i dati a un livello superiore rispetto al singolo noleggio, il numero di righe memorizzate risulta generalmente inferiore rispetto a quello della tabella dei fatti. La presenza di Produttore comporta un incremento dello spazio occupato, ma è giustificata dalla necessità di supportare l'interrogazione relativa ai veicoli Fiat.

2. Definire l’insieme minimale di attributi che permette di identificare le tuple appartenenti alla vista materializzata VM1.

L'insieme è :
                K = {
                    *TipoAlimentazione*,
                    *Produttore*,
                    *Trimstre*,
                    *Anno*,
                    *Regione*
                }

3. Definire i log delle viste materializzate con CREATE MATERIALIZED VIEW LOG per ogni tabella in cui lo si ritiene necessario. Per quali tabelle è utile tenere traccia dei log? Identificare tutte e solo le tabelle necessarie. Inoltre, per ogni tabella identificare tutti e soli gli attributi per i quali è necessario tenere traccia delle variazioni.

In [ ]:
CREATE MATERIALIZED VIEW LOG ON Noleggi
WITH ROWID, SEQUENCE
(
    IdTempo,
    IdPuntoVenditaPartenza,
    IdVeicolo,
    NumNoleggi,
    KmTotali,
    Ricavo,
    OreTotali
)
INCLUDING NEW VALUES;

In [ ]:
CREATE MATERIALIZED VIEW LOG ON Tempo
WITH ROWID, SEQUENCE
(
    IdTempo,
    Trimestre,
    Anno
)
INCLUDING NEW VALUES;

In [ ]:
CREATE MATERIALIZED VIEW LOG ON PuntoVendita
WITH ROWID, SEQUENCE
(
    IdPuntoVendita,
    Regione
)
INCLUDING NEW VALUES;

In [ ]:
CREATE MATERIALIZED VIEW LOG ON Veicolo
WITH ROWID, SEQUENCE
(
    IdVeicolo,
    TipoAlimentazione,
    Produttore
)
INCLUDING NEW VALUES;

I log delle viste materializzate vengono definiti sulle tabelle Noleggi, Tempo, PuntoVendita e Veicolo, in quanto sono le uniche tabelle coinvolte nella definizione della VM1.

Per ciascuna tabella vengono monitorate esclusivamente le variazioni degli attributi utilizzati nella vista materializzata, ovvero:
- gli attributi coinvolti nelle operazioni di JOIN;
- gli attributi dimensionali utilizzati nel SELECT e nel GROUP BY;
- gli attributi utilizzati per il calcolo delle misure aggregate.

Non viene definito alcun log sulla tabella Utente, poiché non partecipa alla costruzione della vista materializzata.

In questo modo si limita lo spazio occupato dai log e si riduce il costo delle operazioni di manutenzione della vista, mantenendo le informazioni necessarie per il suo aggiornamento incrementale.

4. Specificare quali operazioni (ad esempio INSERT su una tabella specifica) causano l'aggiornamento della MATERIALIZED VIEW definita.

La Materialized View VM1 deve essere aggiornata in seguito a operazioni di INSERT, UPDATE e DELETE sulle tabelle di origine Noleggi, Tempo, PuntoVendita e Veicolo, poiché queste tabelle contribuiscono alla definizione della vista attraverso le misure, le condizioni di join e gli attributi di raggruppamento.

Non sono invece necessarie operazioni di aggiornamento in seguito a modifiche sulla tabella Utente, in quanto essa non partecipa alla definizione della Materialized View.

## Aggiornamento tramite trigger

Supponendo che il comando CREATE MATERIALIZED VIEW non sia disponibile, creare le viste materializzate definire nell'esercizio precedente e definite la procedura di aggiornamento a partire dalle modifiche sulla tabella dei fatti create tramite un trigger.

1. Creare la struttura della vista materializzata con CREATE TABLE VM1 (...)

In [ ]:
CREATE TABLE VM1_TRIGGER (
    TipoAlimentazione VARCHAR(20) NOT NULL,
    Trimestre VARCHAR(20) NOT NUll,
    Anno NUMBER(4) NOT NULL,
    Regione VARCHAR(20) NOT NULL,
    Produttore VARCHAR(20) NOT NULL,
    NumNoleggi NUMBER,
    kmTotali NUMBER,
    FatturatoTotale NUMBER,
    OreTotali NUMBER
    PRIMARY KEY (TipoAlimentazione, Trimestre, Anno, Regione, Produttore)
)

2. Specificare un esempio di istruzione per popolare la tabella VM1 con i record
necessari utilizzando l'istruzione INSERT INTO VM1 (...) ( SELECT ... )

In [ ]:
INSERT INTO VM1_TRIGGER (
    TipoAlimentazione, Trimestre, Anno, Regione, Produttore,
    NumNoleggi, kmTotali, FatturatoTotale, OreTotali
    )
    SELECT V.TipoAlimentazione, T.Trimestre, T.Anno, PP.Regione, V.Produttore,
        SUM(N.NumNoleggi), 
        SUM(N.kmTotali), 
        SUM(N.Ricavo), 
        SUM(N.OreTotali) AS OreTotali
    FROM Noleggio N 
    JOIN Tempo T ON T.IdTempo = N.IdTempo
    JOIN PuntoVendita PP ON PP.IdPuntoVendita = N.IdPuntoVendita
    JOIN Veicolo V ON V.IdVeicolo = N.IdVeicolo
    GROUP BY V.TipoAlimentazione, V.Produttore, T.Trimestre, T.Anno, PP.Regione